In [2]:
import pandas as pd
from statsmodels.stats.inter_rater import fleiss_kappa
import numpy as np
from sklearn.metrics import cohen_kappa_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, matthews_corrcoef
import os


"""   

Copyright (c) 2026, Michael Tchuindjang
All rights reserved.

This code was developed as part of a PhD research project in Cybersecurity and Artificial Intelligence, 
supported by a studentship at the University of the West of England (UWE Bristol).

Use of this software is permitted for academic, educational, and research purposes.  
For any commercial use or redistribution, please contact the author for permission.

Disclaimer:
In no event shall the author or UWE be liable for any claim, damages, or other liability arising from the use of this code.

Acknowledgment of the author and the research context is appreciated in any derivative work or publication.


"""

# =========================
# GLOBAL PARAMETERS
# =========================
TEST_DIR = "testing"
#os.makedirs(TEST_DIR, exist_ok=True)

TESTS = ['Test_1', 
         'Test_2', 
         'Test_3', 
         'Test_4']

TESTING_TEST = TESTS[1] # TESTS[0]: Test_1
# Tests/training files are formatted like Test_X_all_models.csv
TESTING_TEST_FILE = TESTING_TEST + '_all_models.csv'
INPUT_FILE = os.path.join(TEST_DIR, TESTING_TEST_FILE)



def jailbreakbench(df):
    
    base_id_list = [2, 10, 13, 57, 58, 59, 60, 62, 66, 70]  # Base_id for jailbreakbench subset
    # Filter
    filtered_df = df[
    (df["turn_depth"] == 1) &
    (df["base_id"].isin(base_id_list))]

    return filtered_df


def cols_ensemble(df, cols, newcol):
    

    
    # Check if all columns exist
    if all(col in df.columns for col in cols):
        # Initialize the ensemble column with NaN
        df[''+newcol] = pd.NA
    
        # Identify rows where all columns have non-NaN values
        valid_rows = df[cols].notna().all(axis=1)
    
        # Perform the ensemble vote only for valid rows
        df.loc[valid_rows, ''+newcol] = (
            (df.loc[valid_rows, cols].sum(axis=1) >= 2).astype(int)
        )
    
        # Convert the entire column to integer if you want to avoid 'object' dtype
        df[newcol] = pd.to_numeric(df[newcol], errors='coerce')
    else:
        print("⚠️ Ensemble not created — one or more required columns are missing.")
        return None
        #sys.exit(1)  # exits the program with error code 1

    return df


def semantic_ensemble():

    df = pd.read_csv(INPUT_FILE)

    new_col_no_chunk = 'prediction_ensemble[no_chunk]_judge' 
    new_col_chunk = 'prediction_ensemble[chunked]_judge' 
    
    # Define the expected model columns
    cols_no_chunk = [
        'prediction_all-MiniLM-L6-v2[no_chunk]_judge',
        'prediction_all-mpnet-base-v2[no_chunk]_judge',
        'prediction_all-roberta-large-v1[no_chunk]_judge'
    ]

    cols_chunk = [
        'prediction_all-MiniLM-L6-v2[chunked]_judge',
        'prediction_all-mpnet-base-v2[chunked]_judge',
        'prediction_all-roberta-large-v1[chunked]_judge'
    ]
    
    df1 = cols_ensemble(df, cols_no_chunk, new_col_no_chunk)
    if df1 is not None:
        final_df = cols_ensemble(df1, cols_chunk, new_col_chunk)
        if final_df is not None:
            final_df.to_csv(INPUT_FILE, index=False)
            print(f"\n✅ Semantic ensemble columns created successfully. Results saved to: {INPUT_FILE}")
        else:
            sys.exit(1)  # exits the program with error code 1
    else:
        sys.exit(1)  # exits the program with error code 1

# =========================
# HELPER: CASE-INSENSITIVE FILTER
# =========================
def normalize_filter(values):
    if isinstance(values, str):
        return [values.lower()]
    return [v.lower() for v in values]

# =========================
# COMPUTE DETAILED METRICS
# =========================
def compute_judge_metrics(df, ground_truth="human_ensemble_judge",
                          model_filter=None,
                          turn_depth_filter=None,
                          tense_filter=None,
                          subtopic_filter=None,
                          source_filter=None,
                          attack_type_filter=None,
                          judge_filter=None,
                          metrics_filter=None):

    df_filtered = df.copy()

    df_filtered["subtopic_norm"] = df_filtered["subtopic"].str.lower()

    # =========================
    # APPLY FILTERS
    # =========================
    if model_filter is not None:
        df_filtered = df_filtered[df_filtered['model_name'].isin([model_filter] if isinstance(model_filter,str) else model_filter)]

    if turn_depth_filter is not None:
        df_filtered = df_filtered[df_filtered['turn_depth'].isin([turn_depth_filter] if isinstance(turn_depth_filter,int) else turn_depth_filter)]

    if tense_filter is not None:
        df_filtered = df_filtered[df_filtered['tense'].isin([tense_filter] if isinstance(tense_filter,str) else tense_filter)]

    if subtopic_filter is not None:
        if isinstance(subtopic_filter, str):
            subtopic_filter = [subtopic_filter]
        subtopic_filter_norm = [s.lower() for s in subtopic_filter]
        df_filtered = df_filtered[df_filtered["subtopic_norm"].isin(subtopic_filter_norm)]

    if source_filter is not None:
        df_filtered = df_filtered[df_filtered['source'].isin([source_filter] if isinstance(source_filter,str) else source_filter)]

    if attack_type_filter is not None:
        df_filtered = df_filtered[df_filtered['source'].isin([attack_type_filter] if isinstance(attack_type_filter,str) else attack_type_filter)]

    # =========================
    # JUDGE SELECTION
    # =========================
    judge_cols = [c for c in df_filtered.columns if c.endswith("_judge") and c != ground_truth]
    #print(judge_cols)
    if judge_filter is not None:
        judge_cols = [j for j in judge_cols if j in judge_filter]
       

    metrics_list = []

    # =========================
    # GROUPING (FIXED)
    # =========================
    grouped = df_filtered.groupby(
        ["model_name", "turn_depth", "tense", "subtopic_norm", "source", "attack_name"]
    )

    for (model_name, turn_depth, tense, subtopic_norm, source, attack_name), df_group in grouped:

        # Use normalized subtopic (consistent output)
        subtopic_label = subtopic_norm

        for judge in judge_cols:
            df_valid = df_group[[ground_truth, judge]].dropna()
            if df_valid.empty:
                continue

            y_true = df_valid[ground_truth]
            y_pred = df_valid[judge]

            tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()

            total_rows = len(df_valid)
            total_success = y_pred.sum()
            asr = (tp + fn) / total_rows * 100

             # FPR and FNR
            fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
            fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

            # ROC AUC
            try:
                roc_auc = roc_auc_score(y_true, y_pred)
            except ValueError:
                roc_auc = None

            # Matthews Correlation Coefficient
            try:
                mcc = matthews_corrcoef(y_true, y_pred)
            except ValueError:
                mcc = None

            metrics_dict = {
                "Judge": judge,
                "Model": model_name,
                "Turn Depth": turn_depth,
                "Tense": tense,
                "Subtopic": subtopic_label,
                "Source": source,
                "Attack Type": attack_name,
                "Total of Success (1s)": total_success,
                "Total of Rows": total_rows,
                "ASR (%)": round(asr,2),
                "Cohen's Kappa": round(cohen_kappa_score(y_true, y_pred),3),
                "Accuracy": round(accuracy_score(y_true, y_pred),3),
                "Precision": round(precision_score(y_true, y_pred, zero_division=0),3),
                "Recall": round(recall_score(y_true, y_pred, zero_division=0),3),
                "F1 (weighted)": round(f1_score(y_true, y_pred, average='weighted', zero_division=0),3),
                "F1 (macro)": round(f1_score(y_true, y_pred, average='macro', zero_division=0),3),
                "TN": tn, "FP": fp, "FN": fn, "TP": tp,
                "FPR": round(fpr,3),
                "FNR": round(fnr,3),
                "ROC AUC": round(roc_auc,3) if roc_auc is not None else None,
                "MCC": round(mcc,3) if mcc is not None else None
            }

            if metrics_filter is not None:
                keep_cols = ["Judge","Model","Turn Depth","Tense","Subtopic","Source", "Attack Type"]
                metrics_dict = {k: v for k, v in metrics_dict.items() if k in metrics_filter or k in keep_cols}

            metrics_list.append(metrics_dict)

    return pd.DataFrame(metrics_list)


# =========================
# AGGREGATION FUNCTION
# =========================
def aggregate_metrics(df_metrics,
                      aggregate_by=["Model","Turn Depth","Tense","Subtopic","Source", "Attack Type"]):
    """
    Aggregates metrics based on selected grouping fields.

    Parameters:
    - df_metrics: output from compute_judge_metrics()
    - aggregate_by: list of columns to group by

    Returns:
    - Aggregated DataFrame
    """

    grouped = df_metrics.groupby(aggregate_by)

    agg_rows = []

    for name, group in grouped:
        if not isinstance(name, tuple):
            name = (name,)

        agg_dict = dict(zip(aggregate_by, name))

        # Sum totals
        if "Total of Success (1s)" in group.columns:
            agg_dict["Total of Success (1s)"] = group["Total of Success (1s)"].sum()
        if "Total of Rows" in group.columns:
            agg_dict["Total of Rows"] = group["Total of Rows"].sum()

        # Average performance metrics
        for m in ["ASR (%)","Accuracy","Recall","F1 (weighted)","F1 (macro)","Precision","Cohen's Kappa","FPR","FNR","MCC"]:
            if m in group.columns:
                agg_dict[m] = round(group[m].mean(), 3)

        # Sum confusion matrix counts
        for m in ["TP","FP","FN","TN"]:
            if m in group.columns:
                agg_dict[m] = group[m].sum()

        agg_rows.append(agg_dict)

    return pd.DataFrame(agg_rows)


# =========================
# FUNCTION TO COMPUTE FLEISS' KAPPA WITH FLEXIBLE AGGREGATION
# =========================
def compute_human_fleiss_kappa(df, human_judges=None,
                         model_filter=None,
                         turn_depth_filter=None,
                         tense_filter=None,
                         subtopic_filter=None,
                         source_filter=None,
                         attack_type_filter=None,
                         aggregate_by=None):
    """
    Compute Fleiss' Kappa for multiple human annotators with flexible aggregation.

    Parameters:
    - df: DataFrame
    - human_judges: list of human judge columns
    - model_filter: case-sensitive
    - turn_depth_filter: int or list
    - tense_filter: case-sensitive
    - subtopic_filter: case-insensitive
    - source_filter: case-sensitive
    - attack_type_filter: case-sensitive
    - aggregate_by: list of columns to group by
        e.g. ["model_name", "turn_depth"]
        default: ["model_name", "turn_depth", "tense", "subtopic", "source", "attack_name"]

    Returns:
    - DataFrame with Fleiss Kappa per group
    """

    if human_judges is None:
        human_judges = [c for c in df.columns if c.startswith("human") and c.endswith("_judge")]

    if aggregate_by is None:
        aggregate_by = ["model_name", "turn_depth", "tense", "subtopic", "source", "attack_name"]

    df_filtered = df.copy()

    # =========================
    # APPLY FILTERS
    # =========================
    if model_filter is not None:
        if isinstance(model_filter, str):
            model_filter = [model_filter]
        df_filtered = df_filtered[df_filtered['model_name'].isin(model_filter)]

    if turn_depth_filter is not None:
        if isinstance(turn_depth_filter, int):
            turn_depth_filter = [turn_depth_filter]
        df_filtered = df_filtered[df_filtered['turn_depth'].isin(turn_depth_filter)]

    if tense_filter is not None:
        if isinstance(tense_filter, str):
            tense_filter = [tense_filter]
        df_filtered = df_filtered[df_filtered['tense'].isin(tense_filter)]

    # Subtopic (case-insensitive)
    if subtopic_filter is not None:
        if isinstance(subtopic_filter, str):
            subtopic_filter = [subtopic_filter]
        subtopic_filter_lower = [s.lower() for s in subtopic_filter]
        df_filtered = df_filtered[df_filtered['subtopic'].str.lower().isin(subtopic_filter_lower)]

    if source_filter is not None:
        if isinstance(source_filter, str):
            source_filter = [source_filter]
        df_filtered = df_filtered[df_filtered['source'].isin(source_filter)]

    if attack_type_filter is not None:
        if isinstance(attack_type_filter, str):
            attack_type_filter = [attack_type_filter]
        df_filtered = df_filtered[df_filtered['attack_name'].isin(attack_type_filter)]

    # =========================
    # GROUPING
    # =========================
    grouped = df_filtered.groupby(aggregate_by)

    results = []

    for group_keys, group_df in grouped:
        # Ensure keys are iterable
        if not isinstance(group_keys, tuple):
            group_keys = (group_keys,)

        group_info = dict(zip(aggregate_by, group_keys))

        # Drop rows with missing human annotations
        df_valid = group_df[human_judges].dropna()
        if df_valid.empty:
            continue

        # Build Fleiss matrix
        n_items = df_valid.shape[0]
        matrix = np.zeros((n_items, 2))

        matrix[:, 0] = (df_valid.values == 0).sum(axis=1)
        matrix[:, 1] = (df_valid.values == 1).sum(axis=1)

        kappa = fleiss_kappa(matrix)

        results.append({
            **group_info,
            "Fleiss Kappa": round(kappa, 3),
            "Num Items": n_items
        })

    return pd.DataFrame(results)


# =========================
# USAGE EXAMPLE
# =========================

#semantic_ensemble()
    
df = pd.read_csv(INPUT_FILE)

#df2 = df.copy()
#df = jailbreakbench(df2)

model_names = ["llama-2-7b-chat","qwen2-7b-instruct","gpt-4o-mini","gemini-2.0-flash"]
tenses = ["present","past"]
turn_depths = [1,2,3]
human_judges_list = ["human_1_judge", "human_2_judge", "human_3_judge"]

subtopics = ["Hacking","Malware","Phishing"]

sources = ["AdvBench","HarmBench","Original"]

attacks = ["crescendo","tempest","mirage"]

# Example: only keep ASR and F1 metrics for detailed per-judge metrics. Full list on metrics_dict definition
#metrics_to_keep = ["F1 (weighted)","Cohen's Kappa","Recall","Precision","FPR","FNR","MCC","TN","TP","FN","FP"]
metrics_to_keep = ["F1 (weighted)","Recall","Precision","FPR","FNR"]

# Filter only specific judges and metrics for detailed computation
selected_judges = ["gpt-4.1_judge","gpt-5.2_judge",
                   "rule-based_judge","llama-guard-3-8b_judge",
                   "prediction_all-MiniLM-L6-v2[no_chunk]_judge","prediction_all-MiniLM-L6-v2[chunked]_judge",
                   "prediction_all-mpnet-base-v2[no_chunk]_judge","prediction_all-mpnet-base-v2[chunked]_judge",
                   "prediction_all-roberta-large-v1[no_chunk]_judge","prediction_all-roberta-large-v1[chunked]_judge",
                   "prediction_ensemble[no_chunk]_judge","prediction_ensemble[chunked]_judge"
                  ]

selected_sources = ["AdvBench","HarmBench"]

model_name = model_names[0]
turn_depth = turn_depths[1]
tense = tenses[0]
subtopic = subtopics[1]
source = sources[1]
attack = attacks[0]

# Step 1: Filter + compute
"""df_metrics = compute_judge_metrics(
    df,
    model_filter=model_name,
    turn_depth_filter=turn_depth,
    tense_filter=tense,
    subtopic_filter=subtopic,
    source_filter=source,
    attack_type_filter=attack,
    judge_filter=selected_judges,
    metrics_filter=metrics_to_keep
)"""
df_metrics = compute_judge_metrics(
    df, judge_filter=selected_judges, source_filter=None, metrics_filter=metrics_to_keep)
#print(df_metrics)
# Step 2: Aggregate flexibly
df_agg = aggregate_metrics(
    df_metrics,
    #aggregate_by=["Judge","Turn Depth","Tense","Attack Type","Source"],
    aggregate_by=["Source","Tense","Judge"]
)

# Save aggregated metrics
df_agg.to_csv(f"{TESTING_TEST}_detailed_agreement_metrics.csv", index=False)

print("✅ Detailed and aggregated metrics saved.")
"""
# Compute Fleiss Kappa for specific filters
df_fleiss_filtered = compute_human_fleiss_kappa(df,
                                                human_judges=human_judges_list,
                                                #model_filter=model_name,
                                                turn_depth_filter=turn_depth,
                                                #tense_filter=tense,
                                                #subtopic_filter=subtopic,
                                                #source_filter=source,
                                                aggregate_by=["turn_depth", "tense", "attack_name"])
df_fleiss_filtered.to_csv("human_fleiss_kappa_filtered.csv", index=False)

# Compute Fleiss Kappa for all human judges across all the subtopics excluded in aggregate_by
df_fleiss = compute_human_fleiss_kappa(df, human_judges=human_judges_list, aggregate_by=["model_name", "turn_depth", "tense", "attack_name", "source"])
df_fleiss.to_csv("human_fleiss_kappa_test2.csv", index=False)
print("✅ Fleiss Kappa computed and saved!")
"""

C:\Users\Michael\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
C:\Users\Michael\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
C:\Users\Michael\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
C:\Users\Michael\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
C:\Users\Michael\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\m

✅ Detailed and aggregated metrics saved.


'\n# Compute Fleiss Kappa for specific filters\ndf_fleiss_filtered = compute_human_fleiss_kappa(df,\n                                                human_judges=human_judges_list,\n                                                #model_filter=model_name,\n                                                turn_depth_filter=turn_depth,\n                                                #tense_filter=tense,\n                                                #subtopic_filter=subtopic,\n                                                #source_filter=source,\n                                                aggregate_by=["turn_depth", "tense", "attack_name"])\ndf_fleiss_filtered.to_csv("human_fleiss_kappa_filtered.csv", index=False)\n\n# Compute Fleiss Kappa for all human judges across all the subtopics excluded in aggregate_by\ndf_fleiss = compute_human_fleiss_kappa(df, human_judges=human_judges_list, aggregate_by=["model_name", "turn_depth", "tense", "attack_name", "source"])\ndf_fleiss.to_csv("h